# GraphML - Assignment 1 | María Sánchez Domínguez

In [18]:
#I have an initial set of codes that ensure the topologicpy installation and version, as well as setting up the libraries. 
#It also shows the initial OBJ file (simple geometry, no apertures yet) to ensure everything is working.
#That is all part 1., part 2. is the assignment graph generation.

## 1.0. - Installation

In [19]:
pip install topologicpy==0.9.19

Note: you may need to restart the kernel to use updated packages.


## 1.1 Import the needed libraries

In [20]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 1.2. Check the TopologicPy Version

In [21]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.19) is OLDER than the latest version (0.9.20) from PyPI. Please consider upgrading to the latest version.


## 1.3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [22]:
renderer = "vscode"

## 1.4. Import the OBJ file

In [23]:
objects = Topology.ByOBJPath(r"assignment1\obj\Assignment1.obj", selfMerge=True)
cells = Topology.Cells(objects[0])
cc = CellComplex.ByCells(cells)
print("Objects is a list")
print(cc)

Objects is a list


## 1.5. Show the geometry

In [24]:
Topology.Show(cc,
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 2.0. ASSIGNMENT: Relational + Aperture Graph

## 2.1. Import geometry OBJ as Cell Complex

In [33]:
objects = Topology.ByOBJPath(r"assignment1\obj\Assignment1.obj", selfMerge=True)
cells = Topology.Cells(objects[0])
building = CellComplex.ByCells(cells)
print(building)

## 2.1. Import apertures (doors, windows) as OBJ (polylines) to generate faces

In [34]:
# Making sure the polylines from the OBJ file are being loaded correctly and as wires
# I found it harder to use the faces directly so I went with an edge extraction->edge to wire->wire to face method

In [36]:
windowFile = Topology.ByOBJPath(r"assignment1\obj\Assignment1_Windows.obj", selfMerge=True)

# Extract wires (the closed polylines)
wires = Topology.Wires(windowFile[0])

# Convert each wire to a face
windows = [Face.ByWire(w) for w in wires]

print(windows)

[<topologic_core.Face object at 0x00000206F1DF5CB0>, <topologic_core.Face object at 0x00000206F1D63DB0>, <topologic_core.Face object at 0x00000206F1D60730>, <topologic_core.Face object at 0x00000206F1D601B0>, <topologic_core.Face object at 0x00000206F1D60030>, <topologic_core.Face object at 0x00000206F1D61270>, <topologic_core.Face object at 0x00000206F1D60870>, <topologic_core.Face object at 0x00000206F4758A70>, <topologic_core.Face object at 0x00000206F475B770>, <topologic_core.Face object at 0x00000206F1D63630>, <topologic_core.Face object at 0x00000206F4759730>, <topologic_core.Face object at 0x00000206F47595F0>, <topologic_core.Face object at 0x00000206F47598B0>, <topologic_core.Face object at 0x00000206F475A2F0>, <topologic_core.Face object at 0x00000206F4759770>]


In [37]:
doorFile = Topology.ByOBJPath(r"assignment1\obj\Assignment1_Doors.obj", selfMerge=True)

# Extract wires (the closed polylines)
wires = Topology.Wires(doorFile[0])

# Convert each wire to a face
doors = [Face.ByWire(w) for w in wires]

print(doors)

[<topologic_core.Face object at 0x00000206F4715CB0>, <topologic_core.Face object at 0x00000206F47BD7F0>, <topologic_core.Face object at 0x00000206F47BCE70>, <topologic_core.Face object at 0x00000206F47BD6F0>, <topologic_core.Face object at 0x00000206F47BF230>, <topologic_core.Face object at 0x00000206F47BEA70>, <topologic_core.Face object at 0x00000206F47BF270>, <topologic_core.Face object at 0x00000206F47BEEB0>, <topologic_core.Face object at 0x00000206F47BDD30>, <topologic_core.Face object at 0x00000206F47BEA30>, <topologic_core.Face object at 0x00000206F47BD8B0>, <topologic_core.Face object at 0x00000206F47BCF70>]


## 2.2. Add door/window faces as Apertures to the Cell Complex

In [38]:
#Adding apertures to the cell complex

building = Topology.AddApertures(building, [doors, windows], exclusive=True, subTopologyType="face")

## 2.3. Generate graphs

### Graph Mapping Strategy

- **Nodes (Vertices):** Each node represents a spatial element, either a cell (room/space) or an aperture (door/window).
- **Edges:** Connections between nodes represent spatial adjacency relationships. An edge exists between two nodes when their corresponding architectural elements physically touch (share a boundary).

**Graph Definitions:**

1. **Cell-to-Cell Graph (g_cells):** 
   - Nodes represent building cells (enclosed volumes/rooms)
   - Edges represent which cells are adjacent to each other
   - Shows the internal topology of the building structure
   - Styled in red (vertices) and black (edges) for clarity
   - Thicker weight for clarity when displayed in combination with g_apertures

2. **Cell-to-Aperture Graph (g_apertures):**
   - Nodes represent both cells AND apertures (doors, windows)
   - Edges represent adjacency between cells and apertures
   - Shows which apertures connect to each cell (all windows + door of a room connected to that room)
   - Styled in cyan (vertices) and yellow (edges) to distinguish from cell-only graph

In [30]:
# I want to display two graphs: one showing cell-to-cell touches and another showing cell-to-aperture touches. 
# I will style them differently to distinguish them visually.

# Graph 1: Cell-to-Cell Touches
g_cells = Graph.BySpatialRelationships(cells, include=["touches"], useInternalVertex=True)

# Graph 2: Cell-to-Aperture Touches
apertures = doors + windows
g_apertures = Graph.BySpatialRelationships(cells + apertures, include=["touches"], useInternalVertex=True)

## 2.4. Graph styling and plotting

In [43]:
# Style cells graph
vertices_cells = Graph.Vertices(g_cells)
for v in vertices_cells:
    d = Dictionary.ByKeysValues(["size", "color"], [20, "red"])
    v = Topology.SetDictionary(v, d)

edges_cells = Graph.Edges(g_cells)
for e in edges_cells:
    d = Dictionary.ByKeysValues(["width", "color"], [8, "black"])
    e = Topology.SetDictionary(e, d)

# Style apertures graph
vertices_apertures = Graph.Vertices(g_apertures)
for v in vertices_apertures:
    d = Dictionary.ByKeysValues(["size", "color"], [10, "cyan"])
    v = Topology.SetDictionary(v, d)

edges_apertures = Graph.Edges(g_apertures)
for e in edges_apertures:
    d = Dictionary.ByKeysValues(["width", "color"], [2, "yellow"])
    e = Topology.SetDictionary(e, d)

# Visualize both graphs together
Topology.Show(building, g_apertures, g_cells,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.2,
              width=600,
              height=600,
              backgroundColor="white",
              renderer=renderer)

## 2.5. Graph Data - Nodes

In [32]:
import pandas as pd

# Print Cell-to-Cell Graph Data
print("CELL-TO-CELL GRAPH (g_cells)")

vertices_data = []
for i, v in enumerate(Graph.Vertices(g_cells)):
    coords = Vertex.Coordinates(v, outputType='xyz', mantissa=2)
    vertices_data.append({"Vertex": i, "X": coords[0], "Y": coords[1], "Z": coords[2]})

print(f"\nVertices ({len(vertices_data)}):")
print(pd.DataFrame(vertices_data).to_string(index=False))

# Print Cell-to-Aperture Graph Data
print("\n")
print("CELL-TO-APERTURE GRAPH (g_apertures)")

vertices_data_apt = []
for i, v in enumerate(Graph.Vertices(g_apertures)):
    coords = Vertex.Coordinates(v, outputType='xyz', mantissa=2)
    vertices_data_apt.append({"Vertex": i, "X": coords[0], "Y": coords[1], "Z": coords[2]})

print(f"\nVertices ({len(vertices_data_apt)}):")
print(pd.DataFrame(vertices_data_apt).to_string(index=False))

CELL-TO-CELL GRAPH (g_cells)

Vertices (16):
 Vertex     X    Y    Z
      0 13.75 1.00 1.25
      1  4.00 6.50 1.25
      2 13.75 4.00 1.25
      3 17.50 2.50 2.50
      4  1.94 1.94 1.25
      5  7.70 2.70 1.25
      6 17.50 1.00 1.25
      7  8.75 1.00 1.25
      8 15.50 2.83 3.75
      9  6.25 1.00 1.25
     10 11.25 4.00 1.25
     11  1.50 7.50 1.25
     12  7.50 5.75 3.75
     13  7.50 6.37 1.25
     14  7.50 7.50 2.50
     15 11.25 1.00 1.25


CELL-TO-APERTURE GRAPH (g_apertures)

Vertices (43):
 Vertex     X    Y    Z
      0  8.75 1.00 1.25
      1 13.75 4.00 1.25
      2 17.50 2.50 2.50
      3 16.50 2.50 1.10
      4 10.50 3.00 1.10
      5  4.00 6.50 1.25
      6  1.50 7.50 1.25
      7  7.50 7.50 2.50
      8 13.75 1.00 1.25
      9  6.90 3.00 1.10
     10 14.50 2.00 1.10
     11  2.50 5.00 1.10
     12  9.50 2.00 1.10
     13  7.50 6.50 1.10
     14  0.00 7.50 1.35
     15 13.75 5.00 1.35
     16  7.70 2.70 1.25
     17  4.32 3.00 1.10
     18 20.00 2.50 1.35
     19 12.0